# Bundle Loading Guide Notebook

Mirrors the [Bundle Loading Guide](https://gongjr0.github.io/SymbolicDSGE/latest/guides/bundle_loading_guide/). Pair with the [authoring notebook](bundle_authoring.ipynb) — the bundle this opens (`experiment-1.sdsge`) is produced there.

In [16]:
import numpy as np

from SymbolicDSGE import load_bundle
from SymbolicDSGE.core.solved_model import SolvedModel
from SymbolicDSGE.estimation.results import MCMCResult
from SymbolicDSGE.estimation.spec import MCMCResultMeta, OptimizationResultMeta
from SymbolicDSGE.monte_carlo import run_pipeline, validate_pipeline_spec

from typing import cast

## Open the bundle

In [17]:
loaded = load_bundle("experiment-1.sdsge")

print("Created by:", loaded.manifest.created_by)
print("Created at:", loaded.manifest.created_at)
print("Format version:", loaded.manifest.sdsge_version)

Created by: experiment-1
Created at: 2026-06-16T12:26:06.366972+00:00
Format version: 1


## Reach the re-solved models

In [18]:
# Cast so type checker doesn't think it can be `None`; completely optional.
reference = cast(SolvedModel, loaded.reference)

# The `if` check narrows the type if casting is not preferred.
if reference is not None:
    print("Stable:", reference.policy.stab == 0)
    print("Eigenvalues:", reference.policy.eig)
    print("A shape:", reference.A.shape)

Stable: True
Eigenvalues: [0.28018451+0.j 0.83      +0.j 0.85      +0.j 2.60451546+0.j
 1.18546572+0.j]
A shape: (5, 5)


In [19]:
T = 20
rng = np.random.default_rng(42)
shocks = {
    "g,z": rng.standard_normal((T, 2)),
}
sim = reference.sim(
    T=T,
    shocks=shocks,
    observables=True,
)
print(sim["Infl"][:5])

[  3.43         8.20387037  10.83531974 -12.17755328   0.91492503]


## Reach the estimation tab

In [20]:
estimation = loaded.estimation

if estimation is not None:
    print("Method:", estimation.spec.method)
    print("Observables:", estimation.spec.observables)
    print("Parameters:", [p.name for p in estimation.spec.parameters])

Method: mcmc
Observables: ['OutGap', 'Infl', 'Rate']
Parameters: ['psi_pi', 'psi_x']


In [21]:
result = estimation.result
if isinstance(result, OptimizationResultMeta):
    print("Point estimate:", result.theta)
    print("Log-posterior:", result.logpost)
elif isinstance(result, MCMCResultMeta):
    print("Acceptance:", round(result.accept_rate, 2))
    print("Draws:", result.n_draws, "burn-in:", result.burn_in)

Acceptance: 0.39
Draws: 1000 burn-in: 200


In [22]:
if estimation.observed is not None:
    print("Observed shape:", estimation.observed.shape)
if estimation.posterior is not None:
    samples = estimation.posterior["samples"]
    print("Posterior mean:", samples.mean(axis=0))

Observed shape: (40, 3)
Posterior mean: [1.91881619 0.71214785]


Reconstruct a live `OptimizationResult` for sampling diagnostics when an MCMC bundle is loaded:

```python
inputs = estimation.spec.to_estimator_inputs()

solver = DSGESolver(loaded.reference.config, loaded.reference.kalman_config)
extras: dict = {}
if inputs.bounds is not None and estimation.spec.method != "mcmc":
    extras["bounds"] = inputs.bounds  # MLE/MAP only

fresh_result = solver.estimate(
    compiled=loaded.reference.compiled,
    y=estimation.observed,
    method=estimation.spec.method,
    estimated_params=inputs.estimated_params,
    theta0=inputs.theta0,
    priors=inputs.priors,
    observables=estimation.spec.observables,
    **extras,
)
print("Re-run result:", type(fresh_result).__name__)
```

### Re-run an estimation from a loaded bundle

`spec.to_estimator_inputs()` lowers the loaded spec into the concrete arguments `DSGESolver.estimate` expects — `Prior` objects built from each `PriorSpec`, parameter initials, and (for MLE/MAP) bounds.

In [23]:
inputs = estimation.spec.to_estimator_inputs()
inputs

EstimatorInputs(estimated_params=['psi_pi', 'psi_x'], theta0={'psi_pi': 2.19, 'psi_x': 0.3}, priors={'psi_pi': Prior(dist=Normal, transform=Identity), 'psi_x': Prior(dist=Normal, transform=Identity)}, bounds=None)

In [24]:
if isinstance(result, MCMCResultMeta) and estimation.posterior is not None:
    mcmc = MCMCResult(
        param_names=result.param_names,
        samples=estimation.posterior["samples"],
        logpost_trace=estimation.posterior["logpost"],
        accept_rate=result.accept_rate,
        n_draws=result.n_draws,
        burn_in=result.burn_in,
        thin=result.thin,
    )
    print(mcmc.hpd_intervals(alpha=0.05))

{'psi_pi': (np.float64(1.490841983236096), np.float64(2.3347453986382276)), 'psi_x': (np.float64(0.37791881078938677), np.float64(1.0960084952086788))}


## Reach the Monte Carlo tab

In [25]:
mc = loaded.mc

if mc is not None:
    print("Pipeline nodes:", [n.id for n in mc.spec.nodes])
    print("Pipeline edges:", [(e.source, e.target) for e in mc.spec.edges])

    if mc.document is not None:
        wire = mc.wire()
        print("Run kind:", wire["kind"])

Pipeline nodes: ['datagen', 'jb_test']
Pipeline edges: [('datagen', 'jb_test')]
Run kind: mc


### Re-run a Monte Carlo pipeline from a loaded bundle

`monte_carlo.run_pipeline` validates the graph against `STEP_CATALOG`, compiles each step, and runs it against the loaded models. No `[ui]` extra is required. Pass `resources=loaded.mc.resources` when the stored spec references raw-data arrays or custom operations.

In [26]:
try:
    ordered = validate_pipeline_spec(
        loaded.mc.spec,
        has_reference=loaded.reference is not None,
        has_dgp=loaded.dgp is not None,
    )
    print("Topological order:", [n.id for n in ordered])
except ValueError as e:
    print(
        "The stored pipeline is invalid for the loaded model roles.\n"
        "See the cell below."
    )

Topological order: ['datagen', 'jb_test']


In [27]:
# The pipeline's simulation datagen needs a DGP. The authoring notebook
# bundles a model under role "dgp", so `loaded.dgp` resolves. If a bundle
# omits `reference` or `dgp`, provide the missing model when re-running.

mc_result = run_pipeline(
    loaded.mc.spec,
    reference=loaded.reference,
    dgp=loaded.dgp,
    n_rep=50,
    fail_fast=True,
    resources=loaded.mc.resources,
)
print("Successful reps:", mc_result.n_successful, "/", mc_result.n_rep)

jb = mc_result.test_summaries["jb_test"]
stat_ci = jb.statistic_confidence_interval(0.95)
pval_ci = jb.pval_confidence_interval(0.95)

list(zip(stat_ci, pval_ci))

Successful reps: 50 / 50


[(np.float64(1.445118149667654), np.float64(0.04347576493189041)),
 (np.float64(3.3898104002811946), np.float64(0.21360231437479654))]

## Reach the simulation prefill

In [28]:
sim_spec = loaded.simulation

if sim_spec is not None:
    print("T:", sim_spec.T)
    print("Observables flag:", sim_spec.observables)
    if sim_spec.shock_generation is not None:
        print("Seed:", sim_spec.shock_generation.seed)
        print("Distribution:", sim_spec.shock_generation.dist)

T: 200
Observables flag: True
Seed: 42
Distribution: norm
